In [8]:
import os
import joblib
os.makedirs('model', exist_ok=True)
joblib.dump(scaler, 'model/scaler.pkl')


['model/scaler.pkl']

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
import joblib

# Load and preprocess data
df = pd.read_csv('Dataset.csv')
df.dropna(inplace=True)

# Encode categorical features
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

X = df.drop('Depression', axis=1)
y = df['Depression']

scaler = StandardScaler()
X = scaler.fit_transform(X)

# Save the scaler for future use
joblib.dump(scaler, 'model/scaler.pkl')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build MLP Model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss=BinaryCrossentropy(), metrics=['accuracy'])
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

# Save model
model.save('model/nn_model.h5')


Epoch 1/50


c:\Users\kunal\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


558/558 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7910 - loss: 0.4493 - val_accuracy: 0.8522 - val_loss: 0.3511
Epoch 2/50
558/558 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8430 - loss: 0.3549 - val_accuracy: 0.8506 - val_loss: 0.3555
Epoch 3/50
558/558 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8458 - loss: 0.3563 - val_accuracy: 0.8490 - val_loss: 0.3513
Epoch 4/50
558/558 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8537 - loss: 0.3387 - val_accuracy: 0.8470 - val_loss: 0.3523
Epoch 5/50
558/558 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8539 - loss: 0.3444 - val_accuracy: 0.8481 - val_loss: 0.3513
Epoch 6/50
558/558 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8507 - loss: 0.3444 - val_accuracy: 0.8495 - val_loss: 0.3557
Epoch 7/50
558/558 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8544 - loss: 0.3385 - val_accuracy: 0.8492 - val_loss: 0.3524
Epoch 8/50
558/558 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8483 - loss: 0.3445 - val_accuracy: 0.8466 - val_

In [10]:
# Fill missing values in 'Financial Stress' with the median
df['Financial Stress'] = df['Financial Stress'].fillna(df['Financial Stress'].median())

# Handle 'Sleep Duration' (map to numerical values)
sleep_duration_map = {
    'Less than 5 hours': 1,
    '5-6 hours': 2,
    '7-8 hours': 3,
    'More than 8 hours': 4
}
df['Sleep Duration'] = df['Sleep Duration'].map(sleep_duration_map)

# Label encoding for binary categorical variables
label_encoder = LabelEncoder()
df['Gender'] = label_encoder.fit_transform(df['Gender'])  # Male: 1, Female: 0
df['Have you ever had suicidal thoughts ?'] = label_encoder.fit_transform(df['Have you ever had suicidal thoughts ?'])  # Yes: 1, No: 0
df['Family History of Mental Illness'] = label_encoder.fit_transform(df['Family History of Mental Illness'])  # Yes: 1, No: 0

# One-Hot Encoding for other categorical variables
df = pd.get_dummies(df, columns=['City', 'Profession', 'Dietary Habits', 'Degree'], drop_first=True)

# Scale numerical features (Age, CGPA, Academic Pressure, Work Pressure, etc.)
scaler = StandardScaler()
numerical_features = ['Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction',
                      'Work/Study Hours', 'Financial Stress']
df[numerical_features] = scaler.fit_transform(df[numerical_features])

# Check for any missing values after preprocessing
print("Missing values after preprocessing:\n", df.isnull().sum())

# Fill missing values if any (using median for all columns)
df.fillna(df.median(), inplace=True)

# Split the data into features (X) and target (y)
X = df.drop('Depression', axis=1)  # Features
y = df['Depression']  # Target variable

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check if there are any missing values in X_train or y_train
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in y_train:", y_train.isnull().sum())

# Train the model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)

# Display the preprocessed data shape
print("Preprocessed Data Shape:", X_train.shape, X_test.shape)


Missing values after preprocessing:
 id                   0
Gender               0
Age                  0
Academic Pressure    0
Work Pressure        0
                    ..
Degree_23            0
Degree_24            0
Degree_25            0
Degree_26            0
Degree_27            0
Length: 108, dtype: int64
Missing values in X_train: 22318
Missing values in y_train: 0
Preprocessed Data Shape: (22318, 107) (5580, 107)


In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Accuracy Score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Classification Report (Precision, Recall, F1-score)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 84.12%

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.77      0.80      2348
           1       0.84      0.89      0.87      3232

    accuracy                           0.84      5580
   macro avg       0.84      0.83      0.84      5580
weighted avg       0.84      0.84      0.84      5580


Confusion Matrix:
[[1815  533]
 [ 353 2879]]


In [12]:
# Save the model
joblib.dump(model, 'random_forest_model.pkl')


['random_forest_model.pkl']

In [14]:
path = 'Dataset.csv'
df1 = pd.read_csv(path)

In [15]:
print(df1['City'].unique())
print(df1['Profession'].unique())
print(df1['Dietary Habits'].unique())
print(df1['Degree'].unique())


['Visakhapatnam' 'Bangalore' 'Srinagar' 'Varanasi' 'Jaipur' 'Pune' 'Thane'
 'Chennai' 'Nagpur' 'Nashik' 'Vadodara' 'Kalyan' 'Rajkot' 'Ahmedabad'
 'Kolkata' 'Mumbai' 'Lucknow' 'Indore' 'Surat' 'Ludhiana' 'Bhopal'
 'Meerut' 'Agra' 'Ghaziabad' 'Hyderabad' 'Vasai-Virar' 'Kanpur' 'Patna'
 'Faridabad' 'Delhi' 'Saanvi' 'M.Tech' 'Bhavna' 'Less Delhi' 'City' '3.0'
 'Less than 5 Kalyan' 'Mira' 'Harsha' 'Vaanya' 'Gaurav' 'Harsh' 'Reyansh'
 'Kibara' 'Rashi' 'ME' 'M.Com' 'Nalyan' 'Mihir' 'Nalini' 'Nandini'
 'Khaziabad']
['Student' 'Civil Engineer' 'Architect' 'UX/UI Designer'
 'Digital Marketer' 'Content Writer' 'Educational Consultant' 'Teacher'
 'Manager' 'Chef' 'Doctor' 'Lawyer' 'Entrepreneur' 'Pharmacist']
['Healthy' 'Moderate' 'Unhealthy' 'Others']
['B.Pharm' 'BSc' 'BA' 'BCA' 'M.Tech' 'PhD' 'Class 12' 'B.Ed' 'LLB' 'BE'
 'M.Ed' 'MSc' 'BHM' 'M.Pharm' 'MCA' 'MA' 'B.Com' 'MD' 'MBA' 'MBBS' 'M.Com'
 'B.Arch' 'LLM' 'B.Tech' 'BBA' 'ME' 'MHM' 'Others']


In [16]:
column_names = X.columns.tolist()

# Print the column names
print(column_names)

['id', 'Gender', 'Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'City_1', 'City_2', 'City_3', 'City_4', 'City_5', 'City_6', 'City_7', 'City_8', 'City_9', 'City_10', 'City_11', 'City_12', 'City_13', 'City_14', 'City_15', 'City_16', 'City_17', 'City_18', 'City_19', 'City_20', 'City_21', 'City_22', 'City_23', 'City_24', 'City_25', 'City_26', 'City_27', 'City_28', 'City_29', 'City_30', 'City_31', 'City_32', 'City_33', 'City_34', 'City_35', 'City_36', 'City_37', 'City_38', 'City_39', 'City_40', 'City_41', 'City_42', 'City_43', 'City_44', 'City_45', 'City_46', 'City_47', 'City_48', 'City_49', 'City_50', 'City_51', 'Profession_1', 'Profession_2', 'Profession_3', 'Profession_4', 'Profession_5', 'Profession_6', 'Profession_7', 'Profession_8', 'Profession_9', 'Profession_10', 'Profession_11', 'Profession_12', 'Profe